In [ ]:
#####| Fixed scenario condition    | What changes across seeds                                                                                |
#####| --------------------------- | -------------------------------------------------------------------------------------------------------- |
#####| `max_inter=0` and `L_var=0` | **No differences** (`inter=0`; `sigmas`/`lambdas` unchanged)                                             |
#####| `max_inter>0` and `L_var=0` | **Only** the interaction matrix `inter` differs                                                          |
#####| `max_inter=0` and `L_var>0` | **Only** `sigmas` / `lambdas` differ (and they affect **t ≥ 4h** since phase 1 uses `sigma=0, lambda=0`) |
#####| `max_inter>0` and `L_var>0` | Both `inter` and `sigmas` / `lambdas` differ                                                             |


# Mechanistic model for data generation under different scenarios
import numpy as np   #for numerical computing. It is used to handle arrays and matrix operations
from scipy.integrate import odeint  #for solving ordinary differential equations (ODEs)

np.random.seed(22)  #This ensures that the random number generation is reproducible, so running the code multiple times will produce the same results.

# =============== define functions ===============
def generate_valid_random(N): ##Generate a random array of size N from a normal distribution such that its mean is less than or equal to 0.1 and its standard deviation is less than or equal to 1.2. If these conditions are not met, the array is regenerated
    while True:
        r = np.random.randn(N)  #randn is a function in NumPy that generates random numbers from a standard normal distribution (i.e., normal distribution with mean 0 and variance 1)
        if abs(np.mean(r)) <= 0.1 and np.std(r) <= 1.2:
            return r
            
def param_generator_lv(N, connectedness, neg_frac, max_neg, max_pos, rnd_seed=None): #Generate an interaction matrix. rnd_seed=None: A seed for random number generation. Every time you run the code without setting a specific seed, the output will be different.
    
    indsPerm = np.random.permutation(N * (N - 1)) #This line creates a random permutation of integers from 0 to N*(N-1)-1, which represents a shuffled order of all possible pairs of species. This is used to randomly assign interactions between species.
    inter_init = np.zeros((N, N - 1)) #Initializes a matrix inter_init of size N x (N-1) filled with zeros. 
    
    neg_count = int(np.ceil(N * (N - 1) * connectedness * neg_frac)) #connectedness: A parameter that controls the overall connectivity between species (how many interactions will exist)
    pos_count = int(np.floor(N * (N - 1) * connectedness * (1 - neg_frac)))
    
    inter_init.flat[indsPerm[:neg_count]] = -np.random.rand(neg_count) * max_neg #generates an array of neg_count random numbers, each drawn independently from a uniform distribution over the interval [0,1)
    inter_init.flat[indsPerm[neg_count:neg_count+pos_count]] = np.random.rand(pos_count) * max_pos
    
    inter = np.zeros((N, N))  ##Initializes a matrix inter of size N x N filled with zeros. This will be the final interaction matrix, where inter[i, j] will store the interaction strength between species i and j
    for i in range(N): #A boolean mask is created that excludes the self-interaction (i.e., mask[i] = False), so species i does not interact with itself.
        mask = np.ones(N, dtype=bool) 
        mask[i] = False 
        inter[i, mask] = inter_init[i, :]
    
    return [inter, inter * (inter < 0), inter * (inter > 0)]  #It returns three matrices: inter: All interactions (both positive and negative). inter * (inter < 0): Only negative interactions. inter * (inter > 0): Only positive interactions.

def core_ode_lv_linparam(x, t, inter, gamma, beta, max_Gs, sigma, lambda_val):
    return x * ((max_Gs * (1 - x) + inter @ x) * (1 - sigma) - lambda_val)

def calculate_growth_rates(x, params, max_Gs, sigma, lambda_val):
    # return G_i-L_i
    return (max_Gs * (1 - x) + params[0] @ x) * (1 - sigma) - lambda_val

def calculate_effective_growth_rates(x, params, max_Gs):
    # return G_i
    return (max_Gs * (1 - x) + params[0] @ x) 

def calculate_lysis_rates(x, params, max_Gs, sigma, lambda_val):
    # return L_i
    return (max_Gs * (1 - x) + params[0] @ x) * sigma + lambda_val

def simulate_timecourse_with_rates(x0, tend, params, max_Gs, sigma, lambda_val):
    t = np.linspace(0, tend, 111)  #This creates an array t of 111 time points, evenly spaced between 0 and tend.
    t_phase1 = t[t < 4]
    t_phase2 = t[t >= 4]

    # Phase 1: growth - first 4h
    y1 = odeint(core_ode_lv_linparam, x0, t_phase1,    ##odeint is used to solve the differential equations over time.
               args=(params[0], params[1], params[2], max_Gs, np.zeros(N), np.zeros(N)),
               atol=1e-9)

    rates1 = np.zeros_like(y1) #This creates an empty array rates1 to store the growth rates for each time point in phase 1.
    for i in range(len(t_phase1)):
        rates1[i] = calculate_growth_rates(y1[i], params, max_Gs, np.zeros(N), np.zeros(N)) 
        
    effective_rates1 = np.zeros_like(y1) 
    for i in range(len(t_phase1)):
        effective_rates1[i] = calculate_effective_growth_rates(y1[i], params, max_Gs)
        
    lysis_rates1 = np.zeros_like(y1) 
    for i in range(len(t_phase1)):
        lysis_rates1[i] = calculate_lysis_rates(y1[i], params, max_Gs, np.zeros(N), np.zeros(N))   

    # Phase 2: growth & lysis - 4h +
    y2 = odeint(core_ode_lv_linparam, y1[-1], t_phase2,
               args=(params[0], params[1], params[2], max_Gs, sigma, lambda_val),
               atol=1e-9)

    rates2 = np.zeros_like(y2)
    for i in range(len(t_phase2)):
        rates2[i] = calculate_growth_rates(y2[i], params, max_Gs, sigma, lambda_val)
    
    effective_rates2 = np.zeros_like(y2)
    for i in range(len(t_phase2)):
        effective_rates2[i] = calculate_effective_growth_rates(y2[i], params, max_Gs)
        
    lysis_rates2 = np.zeros_like(y2)
    for i in range(len(t_phase2)):
        lysis_rates2[i] = calculate_lysis_rates(y2[i], params, max_Gs, sigma, lambda_val)
        
    t
    y = np.vstack([y1, y2]) # Stacks the population data from both phases (y1 and y2) into a single array.
    growth_rates = np.vstack([rates1, rates2]) #Stacks the growth rates from both phases into a single array.
    effective_growth_rates = np.vstack([effective_rates1, effective_rates2])
    lysis_rates = np.vstack([lysis_rates1, lysis_rates2])
    
    return t, y, growth_rates,  effective_growth_rates, lysis_rates

# ====================== Parameter settings ======================
N = 57
x0 = np.ones(N) / (100 * N)
tend = 11 #Simulation time (11 hours).
max_Gs = np.linspace(0.7, 1.5, N) #Maximum growth rates for each species, linearly spaced between 0.7 and 1.5.
sigma = 1.5 * np.ones(N)        # a
lambda_val = 0.1 * np.ones(N)   # b

# lysis variability. Generate two random arrays to introduce variability into the lysis 
random1 = generate_valid_random(N)
random2 = generate_valid_random(N)


connectedness = 1 #density of interactions between species (1 means fully connected).
neg_frac = 0.5
max_neg = 0
max_pos = max_neg * 0.5


# ====================== Initialize output arrays ====================== 
n_time = tend*10+1 # 10 time points per hour
n_species = 57
scenario_params = [{'L_var': lv, 'max_inter': mi} 
                  for lv in [0, 0.1, 0.2] 
                  for mi in [1, 5, 10]]
n_scenarios = len(scenario_params) 

n_seeds = 100  # keep this to define the array size

# Initialize output arrays once for all seeds
outputs = {
    'population_data': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'growth_rates': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'effective_growth_rates': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'lysis_rates': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'relative_abundance': np.zeros((n_time, n_species, n_seeds, n_scenarios))
}

# parameter set
scenario_params = []
for scenario_idx, (L_var, max_inter) in enumerate([(lv, mi) for lv in [0, 0.1, 0.2] for mi in [1, 5, 10]]):  # L_var can be one of [0, 0.1, 0.2]. max_inter can be one of [0, 10, 20]. Generates all 9 combinations of (L_var, max_inter),
    print(f"Running scenario {scenario_idx+1}/{n_scenarios}: L_var={L_var}, max_inter={max_inter}")
    scenario_params.append({'L_var': L_var, 'max_inter': max_inter})

    for seed in range(1, n_seeds + 1):
        np.random.seed(seed)  # set seed

        # impose lysis variability
        sigmas = sigma + L_var * generate_valid_random(N)
        lambdas = lambda_val + L_var * generate_valid_random(N)

        # generate interaction matrix
        current_max_inter = max_Gs[-1] * max_inter if max_inter > 0 else 0
        params = param_generator_lv(N, connectedness, neg_frac, 
                                    max_neg=current_max_inter, 
                                    max_pos=current_max_inter * 0.5)

        # simulate
        t, y, gr, egr, lr = simulate_timecourse_with_rates(x0, tend, params, max_Gs, sigmas, lambdas)

        # store into outputs
        outputs['population_data'][:, :, seed-1, scenario_idx] = y
        outputs['growth_rates'][:, :, seed-1, scenario_idx] = gr
        outputs['effective_growth_rates'][:, :, seed-1, scenario_idx] = egr
        outputs['lysis_rates'][:, :, seed-1, scenario_idx] = lr
        outputs['relative_abundance'][:, :, seed-1, scenario_idx] = y / y.sum(axis=1, keepdims=True)

# Save full dataset
np.savez(f'simulation_results_{n_seeds}seeds.npz',
         time_points=t,
         population_data=outputs['population_data'],
         growth_rates=outputs['growth_rates'],
         effective_growth_rates=outputs['effective_growth_rates'],
         lysis_rates=outputs['lysis_rates'],
         relative_abundance=outputs['relative_abundance'],
         scenario_params=scenario_params)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import csv

# -------------------- config --------------------
n_seeds = 100
npz_path = f"simulation_results_{n_seeds}seeds.npz"
fit_hours   = np.array([4, 5, 6, 8, 10])
eval_hours  = np.array([5, 6, 8])
colors = {5: "#64A9D3", 6: "#216FB0",8: "#054B84"}
scenario_params = [{'L_var': lv, 'max_inter': mi}
                   for lv in [0, 0.1, 0.2]
                   for mi in [1, 5, 10]]
n_scenarios = len(scenario_params)

# ---- font sizes (balanced) ----
FS_SUPTITLE = 16
FS_TITLE    = 12
FS_LABEL    = 12
FS_LEGEND   = 12
FS_TICKS    = 12
FS_TEXTBOX  = 12
SCAT_SIZE   = 14

# -------------------- load --------------------
npz = np.load(npz_path, allow_pickle=True)
t  = npz["time_points"]          # (T,)
pop = npz["population_data"]     # 4D with a seed axis somewhere
gr  = npz["growth_rates"]        # same shape as pop
T = t.size

def to_seed_time_species_scenario(arr, n_seeds, T, n_scenarios):
    """Reorder to (seed, time, species, scenario)."""
    if arr.ndim != 4:
        raise ValueError(f"Need 4D array, got {arr.shape}")
    shape = arr.shape
    idx_seed = [i for i,s in enumerate(shape) if s == n_seeds][0]
    idx_time = [i for i,s in enumerate(shape) if s == T and i != idx_seed][0]
    idx_scen = [i for i,s in enumerate(shape) if s == n_scenarios and i not in (idx_seed, idx_time)][0]
    idx_spec = [ax for ax in range(4) if ax not in (idx_seed, idx_time, idx_scen)][0]
    return np.transpose(arr, (idx_seed, idx_time, idx_spec, idx_scen))

pop4 = to_seed_time_species_scenario(pop, n_seeds, T, n_scenarios)
gr4  = to_seed_time_species_scenario(gr,  n_seeds, T, n_scenarios)
n_seed, _, n_species, _ = pop4.shape

# indices for fit & eval
fit_idx  = np.array([np.argmin(np.abs(t - h)) for h in fit_hours])
eval_idx = np.array([np.argmin(np.abs(t - h)) for h in eval_hours])

# -------------------- plotting --------------------
fig, axes = plt.subplots(3, 3, figsize=(10, 10))   # Use a compact layout
fig.suptitle(
    "True vs fitted net growth (all seeds)\n"
    "Cubic fit on log-biomass using hours [4,5,6,8,10]; derivative at 5 & 6 h",
    y=0.98, fontsize=FS_SUPTITLE
)

with open("R2_per_seed_by_scenario.csv", "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["scenario_index","L_var","max_inter","seed","R2"])

    for scen in range(n_scenarios):
        ax = axes[scen // 3, scen % 3]
        r2_list = []
        plot_true = {h: [] for h in eval_hours}
        plot_fit  = {h: [] for h in eval_hours}

        for sd in range(n_seed):
            y   = pop4[sd, :, :, scen]     # (T, S)
            tru = gr4[sd,  :, :, scen]     # (T, S)
            seed_true_all, seed_fit_all = [], []

            for sp in range(n_species):
                y_fit = np.maximum(y[fit_idx, sp], 1e-8)
                log_y = np.log(y_fit)

                coeffs = np.polyfit(t[fit_idx], log_y, deg=3)
                poly_d = np.poly1d([3*coeffs[0], 2*coeffs[1], coeffs[2]])

                for h, idx in zip(eval_hours, eval_idx):
                    fval = poly_d(t[idx])
                    tval = tru[idx, sp]
                    if np.isfinite(fval) and np.isfinite(tval):
                        plot_true[h].append(tval)
                        plot_fit[h].append(fval)
                        seed_true_all.append(tval)
                        seed_fit_all.append(fval)

            if len(seed_true_all) > 1:
                r2 = r2_score(seed_true_all, seed_fit_all)
                r2_list.append(r2)
                writer.writerow([scen+1, scenario_params[scen]["L_var"],
                                 scenario_params[scen]["max_inter"], sd, r2])

        # scatter all seeds (color by hour)
        for h in eval_hours:
            ax.scatter(plot_true[h], plot_fit[h], alpha=0.35, s=SCAT_SIZE,
                       label=f"{h} h", color=colors[h])

        # identity line & limits
        if plot_true[eval_hours[0]]:
            all_true = np.concatenate([np.array(plot_true[h]) for h in eval_hours])
            all_fit  = np.concatenate([np.array(plot_fit[h])  for h in eval_hours])
            lo = np.floor(min(all_true.min(), all_fit.min()) - 0.1)
            hi = np.ceil(max(all_true.max(), all_fit.max()) + 0.1)
            ax.plot([lo, hi], [lo, hi], "k--", alpha=0.7, linewidth=1)

        # mean ± SD R² text
        if r2_list:
            ax.text(0.04, 0.96,
                    f"R² = {np.mean(r2_list):.3f} ",
                    transform=ax.transAxes, ha="left", va="top",
                    bbox=dict(facecolor="white", alpha=0.85),
                    fontsize=FS_TEXTBOX)
        else:
            ax.text(0.04, 0.96, "R² across seeds: NA",
                    transform=ax.transAxes, ha="left", va="top",
                    bbox=dict(facecolor="white", alpha=0.85),
                    fontsize=FS_TEXTBOX)

        ax.set_title(
            f"Scenario {scen+1}\nL_var={scenario_params[scen]['L_var']}, "
            f"max_inter={scenario_params[scen]['max_inter']}",
            fontsize=FS_TITLE
        )
        ax.set_xlabel("True net growth rate", fontsize=FS_LABEL)
        ax.set_ylabel("Fitted net growth rate", fontsize=FS_LABEL)
        ax.tick_params(axis='both', which='major', labelsize=FS_TICKS)
        ax.grid(alpha=0.2)

        if scen == 0:
            ax.legend(loc="lower right", fontsize=FS_LEGEND)

plt.tight_layout()
plt.savefig(f'True_vs_Fitted_net_growth_allSeeds_fit456810_eval56_balanced_fonts_{n_seeds}seeds.pdf', dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import csv

# -------------------- config --------------------
n_seeds = 100
npz_path = f"simulation_results_{n_seeds}seeds.npz"
fit_hours   = np.array([4, 5, 6, 8, 10])
eval_hours  = np.array([5, 6, 8])
colors = {5: "#64A9D3", 6: "#216FB0", 8: "#054B84"}
scenario_params = [{'L_var': lv, 'max_inter': mi}
                   for lv in [0, 0.1, 0.2]
                   for mi in [1, 5, 10]]
n_scenarios = len(scenario_params)

# Additional settings: control the number of sampled seeds and random seed
n_plot_seeds = 10          # Plot only 10 seeds
rng = np.random.default_rng(42)  # Fixed random seed (can be changed or removed)

# ---- font sizes (balanced) ----
FS_SUPTITLE = 16
FS_TITLE    = 12
FS_LABEL    = 12
FS_LEGEND   = 12
FS_TICKS    = 12
FS_TEXTBOX  = 12
SCAT_SIZE   = 14

# -------------------- load --------------------
npz = np.load(npz_path, allow_pickle=True)
t  = npz["time_points"]          # (T,)
pop = npz["population_data"]     # 4D with a seed axis somewhere
gr  = npz["growth_rates"]        # same shape as pop
T = t.size

def to_seed_time_species_scenario(arr, n_seeds, T, n_scenarios):
    """Reorder to (seed, time, species, scenario)."""
    if arr.ndim != 4:
        raise ValueError(f"Need 4D array, got {arr.shape}")
    shape = arr.shape
    idx_seed = [i for i,s in enumerate(shape) if s == n_seeds][0]
    idx_time = [i for i,s in enumerate(shape) if s == T and i != idx_seed][0]
    idx_scen = [i for i,s in enumerate(shape) if s == n_scenarios and i not in (idx_seed, idx_time)][0]
    idx_spec = [ax for ax in range(4) if ax not in (idx_seed, idx_time, idx_scen)][0]
    return np.transpose(arr, (idx_seed, idx_time, idx_spec, idx_scen))

pop4 = to_seed_time_species_scenario(pop, n_seeds, T, n_scenarios)
gr4  = to_seed_time_species_scenario(gr,  n_seeds, T, n_scenarios)
n_seed, _, n_species, _ = pop4.shape

# Randomly select seed indices for plotting and statistics
seed_indices = np.sort(rng.choice(n_seed, size=n_plot_seeds, replace=False))

# indices for fit & eval
fit_idx  = np.array([np.argmin(np.abs(t - h)) for h in fit_hours])
eval_idx = np.array([np.argmin(np.abs(t - h)) for h in eval_hours])

# -------------------- plotting --------------------
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle(
    "True vs fitted net growth (random subset of seeds)\n"
    "Cubic fit on log-biomass using hours [4,5,6,8,10]; derivative at 5 & 6 h\n"
    f"Plotted seeds: {n_plot_seeds} / {n_seed}",
    y=0.98, fontsize=FS_SUPTITLE
)

# Export only the sampled seeds to CSV
csv_name = f"R2_per_seed_by_scenario_subset_{n_plot_seeds}of{n_seed}.csv"
with open(csv_name, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["scenario_index","L_var","max_inter","seed","R2"])

    for scen in range(n_scenarios):
        ax = axes[scen // 3, scen % 3]
        r2_list = []
        plot_true = {h: [] for h in eval_hours}
        plot_fit  = {h: [] for h in eval_hours}

        # Iterate only over the selected seeds
        for sd in seed_indices:
            y   = pop4[sd, :, :, scen]     # (T, S)
            tru = gr4[sd,  :, :, scen]     # (T, S)
            seed_true_all, seed_fit_all = [], []

            for sp in range(n_species):
                y_fit = np.maximum(y[fit_idx, sp], 1e-8)
                log_y = np.log(y_fit)

                coeffs = np.polyfit(t[fit_idx], log_y, deg=3)
                poly_d = np.poly1d([3*coeffs[0], 2*coeffs[1], coeffs[2]])

                for h, idx in zip(eval_hours, eval_idx):
                    fval = poly_d(t[idx])
                    tval = tru[idx, sp]
                    if np.isfinite(fval) and np.isfinite(tval):
                        plot_true[h].append(tval)
                        plot_fit[h].append(fval)
                        seed_true_all.append(tval)
                        seed_fit_all.append(fval)

            if len(seed_true_all) > 1:
                r2 = r2_score(seed_true_all, seed_fit_all)
                r2_list.append(r2)
                writer.writerow([scen+1, scenario_params[scen]["L_var"],
                                 scenario_params[scen]["max_inter"], sd, r2])

        # scatter all selected seeds (color by hour)
        for h in eval_hours:
            ax.scatter(plot_true[h], plot_fit[h], alpha=0.35, s=SCAT_SIZE,
                       label=f"{h} h", color=colors[h])

        # identity line & limits
        if plot_true[eval_hours[0]]:
            all_true = np.concatenate([np.array(plot_true[h]) for h in eval_hours])
            all_fit  = np.concatenate([np.array(plot_fit[h])  for h in eval_hours])
            lo = np.floor(min(all_true.min(), all_fit.min()) - 0.1)
            hi = np.ceil(max(all_true.max(), all_fit.max()) + 0.1)
            ax.plot([lo, hi], [lo, hi], "k--", alpha=0.7, linewidth=1)
            ax.set_xlim(lo, hi)
            ax.set_ylim(lo, hi)
            ax.set_aspect('equal', adjustable='box')

        # mean R² text over the selected seeds
        if r2_list:
            ax.text(0.04, 0.96,
                    f"R² = {np.mean(r2_list):.3f} ",
                    transform=ax.transAxes, ha="left", va="top",
                    bbox=dict(facecolor="white", alpha=0.85),
                    fontsize=FS_TEXTBOX)
        else:
            ax.text(0.04, 0.96, "R² across seeds: NA",
                    transform=ax.transAxes, ha="left", va="top",
                    bbox=dict(facecolor="white", alpha=0.85),
                    fontsize=FS_TEXTBOX)

        #ax.set_title(
            #f"Scenario {scen+1}\nL_var={scenario_params[scen]['L_var']}, "
            #f"max_inter={scenario_params[scen]['max_inter']}",
            #fontsize=FS_TITLE )
        #ax.set_xlabel("True net growth rate", fontsize=FS_LABEL)
        #ax.set_ylabel("Fitted net growth rate", fontsize=FS_LABEL)
        ax.tick_params(axis='both', which='major', labelsize=FS_TICKS)
        ax.grid(alpha=0.2)

        if scen == 0:
            ax.legend(loc="lower right", fontsize=FS_LEGEND)

plt.tight_layout()
out_png = f"True_vs_Fitted_net_growth_subset_{n_plot_seeds}of{n_seed}_fit456810_eval56.pdf"
plt.savefig(out_png, dpi=300)
plt.show()
